# Weld inspection: training on Kaggle

Runs the training stage of Assignment 2 on a Kaggle GPU and packs everything the
repo needs into `weld_results.zip`. The cells after the setup are `train.ipynb`
from the repo, unchanged, so this is the same pipeline `main.py train` runs.

**Before running**

1. **Settings → Accelerator:** GPU T4 x2 or GPU P100. Training uses one GPU.
2. **Settings → Internet:** on. Kaggle asks for a verified phone number first.
   The notebook downloads the dataset from Roboflow and the COCO weights from
   Ultralytics.
3. **Add-ons → Secrets:** add a secret named `ROBOFLOW_API_KEY` holding your
   Roboflow key, and tick it for this notebook.

Then **Run All**, or **Save Version → Save & Run All** to run it in the
background. Training is capped at about 11 minutes per model, and the whole
notebook takes about 25–30 minutes. Download `weld_results.zip` from the link
in the last cell, or from the Output panel (`/kaggle/working`).

If it stops partway, run it again: finished training runs are reused, not
repeated.

In [ ]:
# Kaggle already has PyTorch with GPU support; only Ultralytics is missing.
# The version matches the repo's uv.lock.
!pip install -q ultralytics==8.4.173

import os

import torch
from kaggle_secrets import UserSecretsClient

os.environ["ROBOFLOW_API_KEY"] = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
os.makedirs("/kaggle/working/weld", exist_ok=True)
os.chdir("/kaggle/working/weld")  # the project folder; the next two cells write its code files
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "none, turn one on under Settings > Accelerator")

In [ ]:
%%writefile pipeline.py
"""
Code shared by train.ipynb and app.py.

Both stages have to agree on which Roboflow dataset version the model is built
on, what its classes are called, and where the weights and results are saved.
Keeping that in one module means the app can never drift from the training run.
"""

import json
import re
import time
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

ROOT = Path(__file__).resolve().parent

# Roboflow Universe dataset, used as published (not forked)
WORKSPACE = "defect-detection-fxtd5"
PROJECT = "weld-defect-lze1x"
VERSION = 1

# The version-1 export calls its classes "0", "1" and "2". The project lists
# 734 Bad-weld, 1109 Good-weld and 1294 Defect labels, and the export's label
# files hold exactly 734 / 1109 / 1294 boxes of class 0 / 1 / 2.
CLASS_NAMES = ["Bad-weld", "Good-weld", "Defect"]
SPLITS = ["train", "valid", "test"]

DATA_DIR = ROOT / "datasets" / f"{PROJECT}-{VERSION}"
WEIGHTS_DIR = ROOT / "weights"
WEIGHTS_PATH = WEIGHTS_DIR / "best.pt"
ARTIFACTS_DIR = ROOT / "artifacts"
RUNS_DIR = ROOT / "runs"

# Written by train.ipynb, read by app.py
SEAMS_CSV = ARTIFACTS_DIR / "seams.csv"
METRICS_CSV = ARTIFACTS_DIR / "detection_metrics.csv"
CONFUSION_CSV = ARTIFACTS_DIR / "detection_confusion_matrix.csv"
OPERATING_POINT_JSON = ARTIFACTS_DIR / "operating_point.json"

# --- The decision and its costs ---------------------------------------------
# One case is one weld seam. The positive class is Bad-weld: the weld needs
# repair. Every figure is in euros per weld; REPORT.md gives the sources.
LABOUR_EUR_PER_HOUR = 40.3    # Eurostat 2025, euro-area industry, incl. employer costs
BASE_RATE = 0.02              # TWI repair-rate survey: mild steel, all products
INSPECT_MINUTES = 5           # one visual check, middle of the 2-8 minute range
REPAIR_MINUTES = 15 + 30 + 10  # grind out, re-weld, re-inspect
REPAIR_MATERIAL_EUR = 10
FIELD_PREMIUM = 0.40          # a repair on site costs 40% more than in the shop
SITE_VISIT_HOURS = 2 * 4      # two people, half a day at the customer's site
COMPLAINT_EUR = 150           # complaint-handling charge per complaint
HUMAN_MISS_RATE = 0.15        # share of bad welds a human inspector misses

INSPECT_EUR = INSPECT_MINUTES * LABOUR_EUR_PER_HOUR / 60
REPAIR_EUR = REPAIR_MINUTES * LABOUR_EUR_PER_HOUR / 60 + REPAIR_MATERIAL_EUR
ESCAPE_EUR = SITE_VISIT_HOURS * LABOUR_EUR_PER_HOUR + REPAIR_EUR * (1 + FIELD_PREMIUM) + COMPLAINT_EUR

# Shop size and the people available for review (placeholders until decided)
WELDS_PER_DAY = 200
INSPECTORS = 1
SHIFT_HOURS = 7.5

DECISIONS = ["pass", "review", "fail"]
THRESHOLDS = np.round(np.arange(0, 1.001, 0.01), 2)
SCORE_CLASSES = ("Bad-weld", "Defect")


def download_dataset(api_key, data_dir=DATA_DIR):
    """
    Download the YOLO export once, then write a data file with the real class names.

    Uses Roboflow's REST API rather than the roboflow package, whose pinned
    dependencies clash with the libraries Kaggle preinstalls. Roboflow builds an
    export the first time it is asked for, so the link is polled until it is ready.
    """
    if not (data_dir / "train" / "labels").exists():
        url = f"https://api.roboflow.com/{WORKSPACE}/{PROJECT}/{VERSION}/yolov11?api_key={api_key}"
        for _ in range(60):
            with urllib.request.urlopen(url, timeout=120) as response:
                link = json.load(response).get("export", {}).get("link")
            if link:
                break
            time.sleep(10)
        else:
            raise RuntimeError("Roboflow did not finish preparing the export within 10 minutes")
        data_dir.mkdir(parents=True, exist_ok=True)
        archive = data_dir / "export.zip"
        urllib.request.urlretrieve(link, archive)
        with zipfile.ZipFile(archive) as z:
            z.extractall(data_dir)
        archive.unlink()
    yaml_path = data_dir / "weld.yaml"
    yaml_path.write_text(
        f"path: '{data_dir}'\n"
        "train: train/images\nval: valid/images\ntest: test/images\n"
        f"names: {CLASS_NAMES}\n"
    )
    return yaml_path


def video_source(image_name):
    """'SampleV2_1' for a frame cut from one of the dataset's videos, else 'other'."""
    match = re.match(r"(SampleV\d_\d)_mp4", image_name)
    return match.group(1) if match else "other"


def load_labels(data_dir=DATA_DIR):
    """
    Read every YOLO label file.

    Returns two data frames: `images` has one row per image with its box count
    per class, and `boxes` has one row per box with its size as a share of the
    image (YOLO coordinates are already relative to image width and height).
    """
    image_rows, box_rows = [], []
    for split in SPLITS:
        for image_path in sorted((data_dir / split / "images").iterdir()):
            label_path = data_dir / split / "labels" / f"{image_path.stem}.txt"
            lines = label_path.read_text().split("\n") if label_path.exists() else []
            counts = dict.fromkeys(CLASS_NAMES, 0)
            for line in filter(None, (l.split() for l in lines)):
                name = CLASS_NAMES[int(line[0])]
                x, y, w, h = map(float, line[1:5])
                counts[name] += 1
                box_rows.append({"split": split, "image": image_path.name, "class": name,
                                 "x": x, "y": y, "w": w, "h": h, "area_pct": 100 * w * h})
            image_rows.append({"split": split, "image": image_path.name,
                               "source": video_source(image_path.name), **counts})
    return pd.DataFrame(image_rows), pd.DataFrame(box_rows)


def dhash(image_path, size=16):
    """
    Difference hash: shrink to (size+1) x size grey pixels and record whether
    each pixel is brighter than its right-hand neighbour. Two photos of the same
    scene differ in only a few of the size*size bits, even after resizing or
    recompression, so a small Hamming distance flags a near-duplicate.
    """
    grey = np.asarray(Image.open(image_path).convert("L").resize((size + 1, size)), dtype=np.int16)
    return (grey[:, 1:] > grey[:, :-1]).flatten()


def nearest_train_distance(images, data_dir=DATA_DIR):
    """For each valid/test image, the Hamming distance to its closest train image."""
    hashes = np.stack([dhash(data_dir / s / "images" / n) for s, n in zip(images["split"], images["image"])])
    is_train = (images["split"] == "train").to_numpy()
    train_hashes = hashes[is_train]
    distance = np.full(len(images), np.nan)
    for i in np.flatnonzero(~is_train):
        distance[i] = (train_hashes != hashes[i]).sum(axis=1).min()
    return pd.Series(distance, index=images.index, name="train_distance")


def predict_boxes(model, image_paths, conf=0.001):
    """Every box the model predicts above `conf`, one row per box, coordinates relative to the image."""
    rows = []
    results = model.predict([str(p) for p in image_paths], conf=conf, verbose=False, stream=True)
    for path, result in zip(image_paths, results):
        for (x, y, w, h), score, cls in zip(result.boxes.xywhn.tolist(), result.boxes.conf.tolist(),
                                            result.boxes.cls.tolist()):
            rows.append({"image": Path(path).name, "class": result.names[int(cls)],
                         "conf": score, "x": x, "y": y, "w": w, "h": h})
    return pd.DataFrame(rows, columns=["image", "class", "conf", "x", "y", "w", "h"])


def _corners(df):
    return np.stack([df["x"] - df["w"] / 2, df["y"] - df["h"] / 2,
                     df["x"] + df["w"] / 2, df["y"] + df["h"] / 2], axis=1)


def seam_scores(seams, predictions, score_classes=SCORE_CLASSES, min_inside=0.5):
    """
    The model's score for each labelled weld seam.

    A predicted box counts towards a seam when at least `min_inside` of its area
    lies inside the seam's box. The seam's score is the highest confidence among
    the counted boxes of `score_classes` (all classes if None), and 0 when there
    are none, so a seam the model finds nothing wrong with is passed at any
    threshold above 0.
    """
    if score_classes is not None:
        predictions = predictions[predictions["class"].isin(score_classes)]
    by_image = dict(tuple(predictions.groupby("image")))
    scores = np.zeros(len(seams))
    for image, index in seams.groupby("image").indices.items():
        pred = by_image.get(image)
        if pred is None:
            continue
        s, p = _corners(seams.iloc[index]), _corners(pred)
        overlap_w = np.clip(np.minimum(s[:, None, 2], p[None, :, 2]) - np.maximum(s[:, None, 0], p[None, :, 0]), 0, None)
        overlap_h = np.clip(np.minimum(s[:, None, 3], p[None, :, 3]) - np.maximum(s[:, None, 1], p[None, :, 1]), 0, None)
        inside = overlap_w * overlap_h / (pred["w"] * pred["h"]).to_numpy()[None, :]
        conf = np.where(inside >= min_inside, pred["conf"].to_numpy()[None, :], 0)
        scores[index] = conf.max(axis=1)
    return pd.Series(scores, index=seams.index, name="score")


def case_costs(repair=REPAIR_EUR, escape=ESCAPE_EUR, inspect=INSPECT_EUR, miss=HUMAN_MISS_RATE):
    """Euro cost of one weld for each (truth, decision) pair."""
    return {
        ("bad", "pass"): escape,
        ("bad", "review"): inspect + (1 - miss) * repair + miss * escape,
        ("bad", "fail"): repair,
        ("good", "pass"): 0.0,
        ("good", "review"): inspect,
        ("good", "fail"): repair,  # a needless repair
    }


def fixed_rates(decision):
    """Rates for a policy that gives every weld the same decision, e.g. 'pass' for doing nothing."""
    return {truth: {d: float(d == decision) for d in DECISIONS} for truth in ("bad", "good")}


def decide(scores, low, high):
    """Pass below `low`, fail at or above `high`, send for human review in between."""
    scores = np.asarray(scores)
    return np.where(scores >= high, "fail", np.where(scores >= low, "review", "pass"))


def decision_rates(scores, is_bad, low, high):
    """Share of the bad welds, and of the good welds, that end up passed, reviewed or failed."""
    decisions, is_bad = decide(scores, low, high), np.asarray(is_bad)
    return {truth: {d: float(np.mean(decisions[mask] == d)) for d in DECISIONS}
            for truth, mask in (("bad", is_bad), ("good", ~is_bad))}


def cost_per_1000(rates, base_rate=BASE_RATE, **costs):
    """
    Expected cost of 1,000 welds at the real-world base rate.

    The rates come from the test welds, where 37% are bad; reweighting them by
    `base_rate` gives what the same model would cost in a shop where far fewer
    welds are bad.
    """
    unit = case_costs(**costs)
    share = {"bad": base_rate, "good": 1 - base_rate}
    return 1000 * sum(share[t] * rates[t][d] * unit[(t, d)] for t in share for d in DECISIONS)


def review_share(rates, base_rate=BASE_RATE):
    return base_rate * rates["bad"]["review"] + (1 - base_rate) * rates["good"]["review"]


def sweep(scores, is_bad, base_rate=BASE_RATE, **costs):
    """Cost per 1,000 welds at every single threshold, with no review band."""
    rows = []
    for t in THRESHOLDS:
        rates = decision_rates(scores, is_bad, t, t)
        rows.append({"threshold": t, "cost per 1000": cost_per_1000(rates, base_rate, **costs),
                     "recall": rates["bad"]["fail"], "false alarm rate": rates["good"]["fail"]})
    return pd.DataFrame(rows)


def review_capacity(welds_per_day=WELDS_PER_DAY, inspectors=INSPECTORS):
    """Largest share of the day's welds the available inspectors can check."""
    return min(1.0, inspectors * SHIFT_HOURS * 60 / INSPECT_MINUTES / welds_per_day)


def best_band(scores, is_bad, max_review=1.0, base_rate=BASE_RATE, **costs):
    """The (low, high) pair with the lowest cost per 1,000 welds whose review share fits `max_review`."""
    best = None
    for i, low in enumerate(THRESHOLDS):
        for high in THRESHOLDS[i:]:
            rates = decision_rates(scores, is_bad, low, high)
            share = review_share(rates, base_rate)
            if share > max_review:
                continue
            cost = cost_per_1000(rates, base_rate, **costs)
            if best is None or cost < best["cost per 1000"]:
                best = {"low": float(low), "high": float(high), "cost per 1000": cost, "review share": share}
    return best


def staffing(share, welds_per_day=WELDS_PER_DAY):
    """Reviews a day, inspector hours a day, and full-time inspectors needed for a review share."""
    reviews = share * welds_per_day
    hours = reviews * INSPECT_MINUTES / 60
    return {"reviews per day": reviews, "inspector hours per day": hours, "inspectors needed": hours / SHIFT_HOURS}


In [ ]:
%%writefile api_key.py
"""
Assignment 2: find and verify your Roboflow API key.

Run it on its own to check your setup before you write any training code:

    uv run python api_key.py

It looks for your key in two places, in this order:

1. The ROBOFLOW_API_KEY environment variable, if you set one.
2. A .env file in this folder. Copy .env.example to .env and paste your
   key in. This is the easy route: you set it once and forget about it.

.env is in .gitignore and must stay there. Your key is a credential: it is
not yours to leak, and a key committed to a public fork is a key you have to go
and revoke.

Your training code should get the key from here rather than reading the
environment directly, so that both routes work:

    from api_key import load_api_key
    rf = Roboflow(api_key=load_api_key())
"""

import json
import os
import sys
import urllib.error
import urllib.request
from pathlib import Path

ROOT = Path(__file__).resolve().parent
ENV_FILE = ROOT / ".env"
VAR = "ROBOFLOW_API_KEY"
VERIFY_URL = "https://api.roboflow.com/?api_key={key}"


def read_env_file(path: Path = ENV_FILE) -> str | None:
    """Pull ROBOFLOW_API_KEY out of a .env file, if there is one.

    Deliberately a tiny parser rather than a dependency: KEY=value per line,
    blank lines and # comments ignored, surrounding quotes stripped.
    """
    if not path.is_file():
        return None
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        name, separator, value = line.partition("=")
        if separator and name.strip() == VAR:
            return value.strip().strip('"').strip("'") or None
    return None


def resolve_api_key() -> tuple[str | None, str]:
    """Return (key, where it came from), without changing anything."""
    from_env = (os.environ.get(VAR) or "").strip()
    if from_env:
        return from_env, f"the {VAR} environment variable"
    from_file = read_env_file()
    if from_file:
        return from_file, ENV_FILE.name
    return None, "nowhere"


def load_api_key(required: bool = True) -> str | None:
    """Return the API key from the environment, else from .env.

    Also puts it into os.environ so libraries that look it up themselves
    (the roboflow SDK does) find it too.
    """
    key, _ = resolve_api_key()
    if key:
        os.environ[VAR] = key
        return key
    if required:
        raise RuntimeError(
            f"No {VAR} found.\n"
            f"  Copy .env.example to .env and paste your key in, or set the\n"
            f"  {VAR} environment variable. Get your key from the Roboflow\n"
            f"  dashboard under Account > Roboflow Keys."
        )
    return None


def masked(key: str) -> str:
    return f"{key[:2]}{'*' * max(len(key) - 4, 4)}{key[-2:]}" if len(key) > 4 else "****"


def verify(key: str, timeout: int = 20) -> tuple[bool, str]:
    """Ask Roboflow whether this key actually works."""
    try:
        with urllib.request.urlopen(VERIFY_URL.format(key=key), timeout=timeout) as response:
            payload = json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as error:
        try:
            detail = json.loads(error.read().decode("utf-8"))["error"]["message"]
        except Exception:
            detail = f"HTTP {error.code}"
        return False, detail
    except urllib.error.URLError as error:
        return False, f"could not reach api.roboflow.com ({error.reason}). Are you online?"
    except Exception as error:  # noqa: BLE001 - surface anything else plainly
        return False, f"{type(error).__name__}: {error}"

    workspace = payload.get("workspace")
    if isinstance(workspace, dict):
        workspace = workspace.get("name") or workspace.get("url")
    return True, str(workspace) if workspace else "key accepted"


def main() -> int:
    key, source = resolve_api_key()
    if not key:
        print(
            f"No API key found.\n\n"
            f"  Copy .env.example to .env and paste your key in:\n\n"
            f"      cp .env.example .env\n\n"
            f"  Get your key from the Roboflow dashboard, under\n"
            f"  Account > Roboflow Keys.",
            file=sys.stderr,
        )
        return 1

    print(f"Found a key in {source}: {masked(key)}")
    print("Checking it against the Roboflow API ...")
    ok, detail = verify(key)
    if not ok:
        print(f"\nThat key did not work: {detail}", file=sys.stderr)
        return 1
    print(f"Key works. Workspace: {detail}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


# Training pipeline: weld inspection

This notebook is the training stage of Assignment 2. `uv run python main.py train`
converts it to a plain script and runs it top to bottom, so every cell must run
without manual steps.

1. Setup
2. The decision and its costs
3. Download the dataset
4. What the dataset contains
5. Finetune
6. Test results against the baselines
7. From boxes to weld decisions
8. Cost against threshold
9. What a person still has to check
10. Save what the app needs

`WELD_QUICK=1 uv run python main.py train` checks that every cell runs, in a
few minutes on a laptop: one epoch of `yolo11n` on 10% of the training images.
Its numbers mean nothing. Without it the notebook runs the full training.

In [ ]:
import json
import os
import shutil

# Keep Ultralytics' per-batch progress bars out of the notebook; the cells
# below print their own summaries. Must be set before ultralytics is imported.
os.environ.setdefault("YOLO_VERBOSE", "False")

import matplotlib
import numpy as np
import pandas as pd

# main.py runs this notebook as a plain script. Outside Jupyter, switch to a
# non-interactive backend so a plot never opens a window and blocks the run.
try:
    get_ipython  # only defined inside Jupyter / IPython
    IN_NOTEBOOK = True
except NameError:
    IN_NOTEBOOK = False
    matplotlib.use("Agg")

import matplotlib.pyplot as plt

import pipeline
from api_key import load_api_key


def show():
    """plt.show() inside Jupyter; just close the figure when run as a script."""
    if IN_NOTEBOOK:
        plt.show()
    else:
        plt.close("all")


# Chart style: fixed categorical colour order, light grid, no top/right frame
plt.rcParams.update({
    "axes.prop_cycle": matplotlib.cycler(color=["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
                                                "#e87ba4", "#008300", "#6250d6", "#e34948"]),
    "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#c3c2b7",
    "axes.grid": True, "axes.axisbelow": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.labelcolor": "#52514e", "xtick.color": "#52514e", "ytick.color": "#52514e",
    "legend.frameon": False,
})

SEED = 42
pipeline.ARTIFACTS_DIR.mkdir(exist_ok=True)
pd.set_option("display.width", 120)
print("running in Jupyter:", IN_NOTEBOOK)

## 2. The decision and its costs

One case is one weld seam, and the positive class is `Bad-weld`: the weld needs
repair. Each weld gets one of three decisions: pass it, send it to a person for
review, or fail it and send it to repair. The reasoning behind the figures is
in REPORT.md. The figures themselves live in `pipeline.py`, so the app prices
decisions exactly as this notebook does.

| Figure | Value | Source |
|---|---|---|
| Labour per hour, incl. employer costs | €40.3 | Eurostat 2025, euro-area industry |
| Share of welds that are bad (base rate) | 2% | TWI repair-rate survey, mild steel, all products |
| Visual check of one weld | 5 min | middle of the 2–8 min range (Therness) |
| Repair in the shop | 15 + 30 + 10 min and €10 of material | grind out, re-weld, re-inspect |
| Bad weld found at the customer | 2 people × 4 h on site, repair +40%, €150 complaint fee | Schuff Steel at NASCC 2024; supplier contract clause |
| Bad welds a human inspector misses | 15% | Harris & Chaney (1969) |
| A good weld failed by mistake | one needless shop repair | |

In [ ]:
unit = pd.Series(pipeline.case_costs()).unstack()[pipeline.DECISIONS]
print(f"one visual check   EUR {pipeline.INSPECT_EUR:7.2f}")
print(f"one shop repair    EUR {pipeline.REPAIR_EUR:7.2f}")
print(f"one escaped weld   EUR {pipeline.ESCAPE_EUR:7.2f}  ({pipeline.ESCAPE_EUR / pipeline.REPAIR_EUR:.1f}x a shop repair)\n")
print("cost of one weld by truth and decision (EUR)\n" + unit.round(2).to_string() + "\n")

REFERENCE = {
    "do nothing": pipeline.cost_per_1000(pipeline.fixed_rates("pass")),
    "a person checks every weld": pipeline.cost_per_1000(pipeline.fixed_rates("review")),
}
for name, cost in REFERENCE.items():
    print(f"{name:28s} EUR {cost:8,.0f} per 1,000 welds")
check_all = pipeline.staffing(1.0)
print(f"\nchecking all {pipeline.WELDS_PER_DAY} welds a day takes {check_all['inspector hours per day']:.1f} "
      f"inspector hours, {check_all['inspectors needed']:.1f} full-time inspectors; "
      f"{pipeline.INSPECTORS} available can check {pipeline.review_capacity():.0%} of the welds")

## 3. Download the dataset

[`defect-detection-fxtd5/weld-defect-lze1x`](https://universe.roboflow.com/defect-detection-fxtd5/weld-defect-lze1x),
version 1, YOLO format, used as published. The export names its classes `0`, `1`
and `2`; `pipeline.download_dataset` writes a data file with the real names
(see `pipeline.CLASS_NAMES` for how they were matched). The download is skipped
when the files are already on disk.

In [ ]:
DATA_YAML = pipeline.download_dataset(load_api_key())
print(DATA_YAML.read_text())

## 4. What the dataset contains

### 4.1 Split sizes and instances per class

In [ ]:
images, boxes = pipeline.load_labels()
names = pipeline.CLASS_NAMES

instances = images.groupby("split")[names].sum().reindex(pipeline.SPLITS)
images_with = (images[names] > 0).groupby(images["split"]).sum().reindex(pipeline.SPLITS)

summary = instances.add_suffix(" boxes").join(images_with.add_suffix(" images"))
summary.insert(0, "images", images["split"].value_counts().reindex(pipeline.SPLITS))
summary.loc["total"] = summary.sum()
print(summary.to_string())

fig, ax = plt.subplots(figsize=(7, 3.5))
instances.T.plot.bar(ax=ax, rot=0)
ax.set_ylabel("boxes")
ax.set_title("Labelled boxes per class and split")
fig.tight_layout()
fig.savefig(pipeline.ARTIFACTS_DIR / "dataset_instances.png", dpi=150)
show()

### 4.2 Cases: images and welds

Two ways to count the cases a decision is made on. Per image: does the photo
contain a bad weld or a marked defect? Per weld: every weld seam is labelled
either `Good-weld` or `Bad-weld`, while `Defect` boxes mark individual flaws,
usually inside a bad weld.

In [ ]:
images["has_problem"] = (images["Bad-weld"] + images["Defect"]) > 0
cases = images.groupby("split")["has_problem"].agg(images="size", with_problem="sum")
cases = cases.reindex(pipeline.SPLITS)
cases["good welds only"] = cases["images"] - cases["with_problem"]
cases["problem share"] = (cases["with_problem"] / cases["images"]).round(2)
print("per image\n" + cases.to_string() + "\n")

welds = instances[["Good-weld", "Bad-weld"]].copy()
welds["bad share"] = (welds["Bad-weld"] / welds.sum(axis=1)).round(2)
print("per weld seam\n" + welds.to_string() + "\n")

defect_only = images[(images["Defect"] > 0) & (images["Bad-weld"] + images["Good-weld"] == 0)]
print("images with Defect boxes but no weld box:",
      defect_only["split"].value_counts().reindex(pipeline.SPLITS).to_dict())

### 4.3 Object size

Box area as a share of the image. All images are 640 x 640, so a box under
0.25% of the image is smaller than 32 x 32 pixels, the usual cut-off for a
"small" object.

In [ ]:
from PIL import Image

image_sizes = {Image.open(pipeline.DATA_DIR / s / "images" / n).size
               for s, n in zip(images["split"], images["image"])}
print("image sizes:", image_sizes)

size_stats = boxes.groupby("class")["area_pct"].describe(percentiles=[0.25, 0.5, 0.75]).reindex(names)
size_stats["small share"] = boxes.groupby("class")["area_pct"].apply(lambda a: (a < 0.25).mean())
print("box area, % of image")
print(size_stats[["count", "min", "25%", "50%", "75%", "max", "small share"]].round(2).to_string())

fig, ax = plt.subplots(figsize=(7, 3.5))
bins = np.logspace(np.log10(boxes["area_pct"].min()), 2, 30)
for name in names:
    ax.hist(boxes.loc[boxes["class"] == name, "area_pct"], bins=bins, histtype="step", linewidth=2, label=name)
ax.set_xscale("log")
ax.set_xlabel("box area, % of image (log scale)")
ax.set_ylabel("boxes")
ax.set_title("Object size by class")
ax.legend()
fig.tight_layout()
fig.savefig(pipeline.ARTIFACTS_DIR / "dataset_box_sizes.png", dpi=150)
show()

### 4.4 Where the images come from

Many file names show the image is a frame cut from one of six videos
(`SampleV1_1` to `SampleV3_2`). Frames of the same video look alike, so a frame
in the test split may have a near-twin in the train split. The check below
compares every valid and test image with its closest train image using a
difference hash (`pipeline.dhash`); 10 or fewer differing bits out of 256 means
the two are near-duplicates.

In [ ]:
print("images by source\n" + pd.crosstab(images["source"], images["split"])
      .reindex(columns=pipeline.SPLITS).to_string() + "\n")

images["train_distance"] = pipeline.nearest_train_distance(images)
images["near_duplicate"] = images["train_distance"] <= 10
for split in ["valid", "test"]:
    part = images[images["split"] == split]
    print(f"{split}: {len(part)} images, {int((part['train_distance'] == 0).sum())} identical to a "
          f"train image, {int(part['near_duplicate'].sum())} near-duplicates "
          f"({int(part.loc[part['source'] != 'other', 'near_duplicate'].sum())} of them video frames)")

## 5. Finetune

Both `yolo11n` and `yolo11s` start from their COCO weights and are finetuned
with the same settings:

- **Time limit:** each model gets at most 0.18 hours (about 11 minutes) of
  training, which overrides the epoch count, so the whole notebook fits in 30
  minutes on a Kaggle GPU. Ultralytics fits its learning-rate schedule to the
  epochs that fit in the time.
- **Early stopping:** training also stops when validation mAP50-95 has not
  improved for 20 epochs, and Ultralytics keeps the best epoch.
- **Speed settings:** images are cached in memory, batches hold 32 images, and
  the GPU is allowed non-deterministic kernels. Runs are faster but not
  bit-for-bit repeatable.

The model with the higher mAP50-95 on the **valid** split is kept and copied to
`weights/best.pt`; the test split plays no part in the choice.

A finished run is reused rather than retrained when `runs/<model>/done.json`
shows it was trained with exactly these settings. That lets the evaluation below
be re-run after changing a cost figure. A fresh copy of the repo has no `runs/`,
so it always trains.

In [ ]:
import torch
from ultralytics import YOLO

QUICK = os.environ.get("WELD_QUICK") == "1"  # a test that the cells run, not a real model
VARIANTS = ["yolo11n"] if QUICK else ["yolo11n", "yolo11s"]
DEVICE = 0 if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
TRAIN_ARGS = {"epochs": 100, "time": 0.18, "patience": 20, "imgsz": 640, "batch": 32, "workers": 4,
              "cache": "ram", "seed": SEED, "deterministic": False}
if QUICK:
    TRAIN_ARGS.update(epochs=1, time=None, batch=4, workers=2, cache=False, fraction=0.1)
PRETRAINED = pipeline.RUNS_DIR / "pretrained"
PRETRAINED.mkdir(parents=True, exist_ok=True)
print(f"device {DEVICE}, models {VARIANTS}, {TRAIN_ARGS}")

candidates = []
for variant in VARIANTS:
    run = pipeline.RUNS_DIR / (f"{variant}-quick" if QUICK else variant)
    done = run / "done.json"
    if not QUICK and done.exists() and json.loads(done.read_text()) == TRAIN_ARGS:
        print(f"{variant}: reusing the finished run in {run.relative_to(pipeline.ROOT)}")
    else:
        YOLO(PRETRAINED / f"{variant}.pt").train(data=str(DATA_YAML), device=DEVICE, project=str(pipeline.RUNS_DIR),
                                                 name=run.name, exist_ok=True, **TRAIN_ARGS)
        done.write_text(json.dumps(TRAIN_ARGS))
    val = YOLO(run / "weights" / "best.pt").val(data=str(DATA_YAML), split="val", device=DEVICE, plots=False,
                                                 project=str(pipeline.RUNS_DIR), name=f"{run.name}-val", exist_ok=True)
    candidates.append({"model": variant, "run": run.name, "epochs run": len(pd.read_csv(run / "results.csv")),
                       "val AP50": val.box.map50, "val mAP50-95": val.box.map})

candidates = pd.DataFrame(candidates)
best = candidates.loc[candidates["val mAP50-95"].idxmax()]
CHOSEN, chosen_run = best["model"], pipeline.RUNS_DIR / best["run"]
shutil.copy(chosen_run / "weights" / "best.pt", pipeline.WEIGHTS_PATH)
if (chosen_run / "results.png").exists():
    shutil.copy(chosen_run / "results.png", pipeline.ARTIFACTS_DIR / "training_curves.png")
print(candidates.round(3).to_string(index=False))
print(f"\nkept {CHOSEN}, saved to {pipeline.WEIGHTS_PATH.relative_to(pipeline.ROOT)}")

## 6. Test results against the baselines

All three are scored on the same 84 test images.

- **Finetuned model:** Ultralytics' validator. Per-class precision and recall
  are at the confidence that maximises F1; AP50 and mAP50-95 cover all
  confidences.
- **COCO model, zero-shot:** its 80 classes include nothing like a weld, so it
  cannot score above zero on these classes. The cell lists what it predicts
  instead.
- **Trivial baseline:** predicts nothing, so its recall and AP are 0.

In [ ]:
model = YOLO(pipeline.WEIGHTS_PATH)
test = model.val(data=str(DATA_YAML), split="test", device=DEVICE, plots=True,
                 project=str(pipeline.RUNS_DIR), name="test", exist_ok=True)

finetuned = f"finetuned {CHOSEN}"
rows = []
for i, c in enumerate(test.ap_class_index):
    p, r, ap50, ap = test.box.class_result(i)
    rows.append({"model": finetuned, "class": names[c], "precision": p, "recall": r, "AP50": ap50, "mAP50-95": ap})
rows.append({"model": finetuned, "class": "all", "precision": test.box.mp, "recall": test.box.mr,
             "AP50": test.box.map50, "mAP50-95": test.box.map})
for baseline in [f"COCO {CHOSEN}, zero-shot", "predicts nothing"]:
    rows += [{"model": baseline, "class": name, "precision": np.nan, "recall": 0.0, "AP50": 0.0, "mAP50-95": 0.0}
             for name in names + ["all"]]
metrics = pd.DataFrame(rows)
print(metrics.round(3).to_string(index=False))

# Predictions on every valid and test image, used from section 7 on
eval_images = images[images["split"].isin(["valid", "test"])].reset_index(drop=True)
assert eval_images["image"].is_unique
paths = [pipeline.DATA_DIR / s / "images" / n for s, n in zip(eval_images["split"], eval_images["image"])]
coco = YOLO(PRETRAINED / f"{CHOSEN}.pt")
predictions = {"finetuned": pipeline.predict_boxes(model, paths), "coco": pipeline.predict_boxes(coco, paths)}

test_images = set(eval_images.loc[eval_images["split"] == "test", "image"])
coco_test = predictions["coco"][predictions["coco"]["image"].isin(test_images) & (predictions["coco"]["conf"] >= 0.25)]
print(f"\nCOCO model on the test images: {len(coco_test)} boxes at confidence 0.25 or more, by class")
print(coco_test["class"].value_counts().head(8).to_string())

### 6.1 Confusion matrix

Test boxes, counting only predictions at a confidence of 0.25 or more, matched
to a labelled box at an IoU of 0.45. Ultralytics builds this matrix at the
validator's confidence, which is 0.001 for the AP numbers above, so every faint
box would count as an error; a second pass at 0.25 gives a matrix that reflects
what the model actually reports. `background` as the true class is a box the
model predicted where nothing is labelled; as the predicted class, it is a
labelled box the model missed.

In [ ]:
# plots=True is what makes Ultralytics fill the confusion matrix at all
at_025 = model.val(data=str(DATA_YAML), split="test", device=DEVICE, conf=0.25, plots=True,
                   project=str(pipeline.RUNS_DIR), name="test-conf025", exist_ok=True)
labels = names + ["background"]
cm = pd.DataFrame(at_025.confusion_matrix.matrix.T.astype(int), index=labels, columns=labels)
cm.index.name, cm.columns.name = "true", "predicted"
print(cm.to_string())

fig, ax = plt.subplots(figsize=(5.2, 4.4))
values = cm.to_numpy()
ax.imshow(values, cmap="Blues")
ax.grid(False)
ax.set_xticks(range(len(labels)), labels, rotation=30, ha="right")
ax.set_yticks(range(len(labels)), labels)
for (i, j), v in np.ndenumerate(values):
    ax.text(j, i, v, ha="center", va="center", color="white" if v > values.max() / 2 else "#0b0b0b")
ax.set_xlabel("predicted")
ax.set_ylabel("true")
ax.set_title(f"Test boxes, {finetuned}")
fig.tight_layout()
fig.savefig(pipeline.ARTIFACTS_DIR / "detection_confusion_matrix.png", dpi=150)
show()

## 7. From boxes to weld decisions

The decision is made per weld, so the detector's boxes are turned into one score
per labelled weld seam (`pipeline.seam_scores`):

- A predicted `Bad-weld` or `Defect` box counts towards a seam when at least
  half of it lies inside the seam's box.
- The seam's score is the highest confidence among those boxes, and 0 when there
  are none.

`Defect` boxes count because, in the labels, they sit inside bad welds; the
first line below checks that. The COCO model gets the most generous reading:
any of its boxes, whatever the class, counts as evidence of a bad weld.

Images with `Defect` boxes but no weld box have no seam, so they are not cases.

In [ ]:
seams = boxes[boxes["class"].isin(["Bad-weld", "Good-weld"])].reset_index(drop=True)
labelled_defects = boxes[boxes["class"] == "Defect"].assign(conf=1.0)
contains_defect = pipeline.seam_scores(seams, labelled_defects) > 0
print("seams with a labelled Defect box inside, all splits:",
      contains_defect.groupby(seams["class"]).agg(["sum", "size"]).to_dict("index"), "\n")

seams = seams[seams["split"].isin(["valid", "test"])].merge(
    images[["split", "image", "source", "near_duplicate"]], on=["split", "image"])
seams["is_bad"] = seams["class"] == "Bad-weld"
seams["score"] = pipeline.seam_scores(seams, predictions["finetuned"]).to_numpy()
seams["score_coco"] = pipeline.seam_scores(seams, predictions["coco"], score_classes=None).to_numpy()
print(pd.crosstab(seams["split"], seams["class"]).reindex(["valid", "test"]).to_string(), "\n")

test_seams = seams[seams["split"] == "test"]
fig, ax = plt.subplots(figsize=(7, 3.5))
bins = np.linspace(0, 1, 21)
for cls, color in [("Good-weld", "#1baf7a"), ("Bad-weld", "#e34948")]:
    ax.hist(test_seams.loc[test_seams["class"] == cls, "score"], bins=bins, histtype="step", linewidth=2,
            color=color, label=cls)
ax.set_xlabel("seam score (highest Bad-weld or Defect confidence inside the seam)")
ax.set_ylabel("test welds")
ax.set_title("How the finetuned model scores good and bad welds")
ax.legend()
fig.tight_layout()
fig.savefig(pipeline.ARTIFACTS_DIR / "seam_scores.png", dpi=150)
show()

## 8. Cost against threshold

With one threshold and no review, a weld fails when its score is at or above the
threshold and passes otherwise. For every threshold from 0 to 1, the shares of
bad and good welds that pass or fail come from the labelled welds, then get
reweighted to the 2% base rate (`pipeline.cost_per_1000`).

The threshold is chosen on the **valid** split, which has more welds than test,
and the test split then reports what it costs. That keeps the test cost honest:
it was not tuned on the welds it is measured on.

In [ ]:
splits = {s: seams[seams["split"] == s] for s in ["valid", "test"]}
curves = {(s, col): pipeline.sweep(df[col], df["is_bad"]) for s, df in splits.items() for col in ["score", "score_coco"]}
valid_curve, test_curve = curves[("valid", "score")], curves[("test", "score")]
THRESHOLD = float(valid_curve.loc[valid_curve["cost per 1000"].idxmin(), "threshold"])


def plot_cost_curve(band=None):
    """The sweep, with the reference costs; section 9 calls it again to add the review band."""
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(test_curve["threshold"], test_curve["cost per 1000"], color="#2a78d6", linewidth=2, label=f"{finetuned}, test")
    ax.plot(valid_curve["threshold"], valid_curve["cost per 1000"], color="#2a78d6", linestyle="--", label=f"{finetuned}, valid")
    coco_curve = curves[("test", "score_coco")]
    ax.plot(coco_curve["threshold"], coco_curve["cost per 1000"], color="#eb6834", linewidth=2, label="COCO zero-shot, test")
    references = list(REFERENCE.items())
    colors = ["#52514e", "#1baf7a"]
    if band is not None:
        low, high, cost = band
        references.append((f"model + review, {low:.2f} to {high:.2f}, test", cost))
        colors.append("#b07800")
    for (name, cost), color in zip(references, colors):
        ax.axhline(cost, color=color, linestyle=":" if color != "#b07800" else "-", linewidth=1.5)
        ax.text(0.99, cost, name, ha="right", va="bottom", color=color, fontsize=9, transform=ax.get_yaxis_transform())
    ax.axvline(THRESHOLD, color="#e34948", linewidth=1)
    ax.text(THRESHOLD, 0.02, f" chosen {THRESHOLD:.2f}", color="#e34948", fontsize=9, transform=ax.get_xaxis_transform())
    ax.set_ylim(0, 1.3 * REFERENCE["do nothing"])
    ax.set_xlabel("confidence threshold (fail at or above)")
    ax.set_ylabel("EUR per 1,000 welds")
    ax.set_title(f"Expected cost at a {pipeline.BASE_RATE:.0%} base rate")
    ax.legend(loc="upper left")
    fig.tight_layout()
    fig.savefig(pipeline.ARTIFACTS_DIR / "cost_curve.png", dpi=150)
    show()


plot_cost_curve()

print(f"threshold chosen on valid: {THRESHOLD:.2f}")
for s in splits:
    row = curves[(s, "score")].set_index("threshold").loc[THRESHOLD]
    print(f"  {s:5s}: recall {row['recall']:.2f}, false alarm rate {row['false alarm rate']:.3f}, "
          f"EUR {row['cost per 1000']:,.0f} per 1,000 welds")
print(f"the cheapest threshold on test itself would be {test_curve.loc[test_curve['cost per 1000'].idxmin(), 'threshold']:.2f} "
      f"(EUR {test_curve['cost per 1000'].min():,.0f})")

### 8.1 Which figures change the answer

The same choice, threshold on valid and cost on test, repeated for other
escaped-weld costs and base rates. €200 is a bad weld caught at the next step in
the shop; €4,700 is the "100" end of the 1-10-100 rule, a weld that fails in
service. The second cell finds the escaped-weld cost at which the cheapest of
the three options switches.

In [ ]:
def model_alone_cost(base_rate=pipeline.BASE_RATE, **costs):
    """Test cost at the threshold that is cheapest on valid, under the given figures."""
    v = pipeline.sweep(splits["valid"]["score"], splits["valid"]["is_bad"], base_rate, **costs)
    t = float(v.loc[v["cost per 1000"].idxmin(), "threshold"])
    rates = pipeline.decision_rates(splits["test"]["score"], splits["test"]["is_bad"], t, t)
    return t, pipeline.cost_per_1000(rates, base_rate, **costs)


rows = []
for escape in [200, pipeline.ESCAPE_EUR, 4700]:
    for base_rate in [0.01, 0.02, 0.03]:
        t, cost = model_alone_cost(base_rate, escape=escape)
        rows.append({"escaped weld EUR": escape, "base rate": base_rate, "best threshold": t, "model alone": cost,
                     "do nothing": pipeline.cost_per_1000(pipeline.fixed_rates("pass"), base_rate, escape=escape),
                     "person checks all": pipeline.cost_per_1000(pipeline.fixed_rates("review"), base_rate, escape=escape)})
sensitivity = pd.DataFrame(rows)
sensitivity["cheapest"] = sensitivity[["model alone", "do nothing", "person checks all"]].idxmin(axis=1)
print(sensitivity.round(2).to_string(index=False))

In [ ]:
previous = None
print(f"cheapest option at a {pipeline.BASE_RATE:.0%} base rate as the escaped-weld cost rises:")
for escape in range(50, 6001, 10):
    options = {"model alone": model_alone_cost(escape=escape)[1],
               "do nothing": pipeline.cost_per_1000(pipeline.fixed_rates("pass"), escape=escape),
               "person checks all": pipeline.cost_per_1000(pipeline.fixed_rates("review"), escape=escape)}
    cheapest = min(options, key=options.get)
    if cheapest != previous:
        print(f"  from EUR {escape:>5,}: {cheapest}")
        previous = cheapest

## 9. What a person still has to check

Two thresholds: fail at or above `high`, pass below `low`, and send everything
in between to a person. A reviewed weld costs one visual check, and the
inspector still misses 15% of the bad ones.

The pair is chosen on the valid split: every combination on the 0.01 grid,
keeping only those whose review load fits the inspectors available
(`pipeline.review_capacity`), then the cheapest. The test split reports the
result against the three alternatives.

In [ ]:
capacity = pipeline.review_capacity()
band = pipeline.best_band(splits["valid"]["score"], splits["valid"]["is_bad"], max_review=capacity)
unlimited = pipeline.best_band(splits["valid"]["score"], splits["valid"]["is_bad"])
LOW, HIGH = band["low"], band["high"]
print(f"{pipeline.INSPECTORS} inspector(s) can check {capacity:.0%} of {pipeline.WELDS_PER_DAY} welds a day")
print(f"band chosen on valid: pass below {LOW:.2f}, review from {LOW:.2f} to {HIGH:.2f}, fail at {HIGH:.2f} or above")
print(f"with no limit on reviewers it would be {unlimited['low']:.2f} to {unlimited['high']:.2f}, "
      f"reviewing {unlimited['review share']:.0%}\n")

test_seams = splits["test"]
policies = {
    "do nothing": pipeline.fixed_rates("pass"),
    "a person checks every weld": pipeline.fixed_rates("review"),
    f"model alone, threshold {THRESHOLD:.2f}": pipeline.decision_rates(test_seams["score"], test_seams["is_bad"], THRESHOLD, THRESHOLD),
    f"model + review, {LOW:.2f} to {HIGH:.2f}": pipeline.decision_rates(test_seams["score"], test_seams["is_bad"], LOW, HIGH),
}
rows = []
for name, rates in policies.items():
    share = pipeline.review_share(rates)
    rows.append({"policy": name, "EUR per 1,000 welds": pipeline.cost_per_1000(rates), "sent to review": share,
                 "bad welds caught": rates["bad"]["fail"] + rates["bad"]["review"] * (1 - pipeline.HUMAN_MISS_RATE),
                 "good welds failed": rates["good"]["fail"], **pipeline.staffing(share)})
comparison = pd.DataFrame(rows).set_index("policy")
print(comparison.round(2).to_string())

decisions = pipeline.decide(test_seams["score"], LOW, HIGH)
counts = pd.crosstab(test_seams["class"], decisions).reindex(columns=pipeline.DECISIONS, fill_value=0)
print("\ntest welds by decision, with the band\n" + counts.rename_axis(columns="decision").to_string())

plot_cost_curve((LOW, HIGH, comparison["EUR per 1,000 welds"].iloc[-1]))

### 9.1 Which figures change the band

The scenarios of section 8.1 again, now with the band re-chosen on the valid
split for each one, under the same limit on reviewers. The cheapest column
compares all four options on the test split.

In [ ]:
band_rows = []
for escape, base_rate in zip(sensitivity["escaped weld EUR"], sensitivity["base rate"]):
    b =pipeline.best_band(splits["valid"]["score"], splits["valid"]["is_bad"], max_review=capacity,
                           base_rate=base_rate, escape=escape)
    rates = pipeline.decision_rates(test_seams["score"], test_seams["is_bad"], b["low"], b["high"])
    band_rows.append({"band": f"{b['low']:.2f} to {b['high']:.2f}", "sent to review": pipeline.review_share(rates, base_rate),
                      "model + review": pipeline.cost_per_1000(rates, base_rate, escape=escape)})
with_band = pd.concat([sensitivity.drop(columns="cheapest"), pd.DataFrame(band_rows)], axis=1)
options = ["model + review", "model alone", "do nothing", "person checks all"]
with_band["cheapest"] = with_band[options].idxmin(axis=1)
print(with_band.round(2).to_string(index=False))

### 9.2 Where the band goes wrong

The same test decisions split by where the image came from (a frame of one of
the six videos, or anything else) and by whether it is a near-duplicate of a
train image. A gap between the groups means the headline number leans on
images that resemble the training set.

The figure shows the bad welds the band passes, lowest score first, and the
good welds the model scores highest, which are the ones it sends to review or
fails.

In [ ]:
test_seams = test_seams.assign(decision=decisions,
                               origin=np.where(test_seams["source"] == "other", "other images", "video frames"),
                               duplicate=np.where(test_seams["near_duplicate"], "near-duplicate", "no near-duplicate"))
for column in ["origin", "duplicate"]:
    bad = test_seams[test_seams["is_bad"]].groupby(column)["decision"]
    good = test_seams[~test_seams["is_bad"]].groupby(column)["decision"]
    table = pd.DataFrame({"bad welds": bad.size(), "bad passed": bad.apply(lambda d: (d == "pass").mean()),
                          "good welds": good.size(), "good failed": good.apply(lambda d: (d == "fail").mean())})
    print(table.round(2).to_string(), "\n")

missed = test_seams[test_seams["is_bad"] & (test_seams["decision"] == "pass")].sort_values("score").head(4)
confusing = test_seams[~test_seams["is_bad"] & (test_seams["decision"] != "pass")].sort_values("score", ascending=False).head(4)
fig, axes = plt.subplots(2, 4, figsize=(12, 6.4))
for row_axes, cases, kind in [(axes[0], missed, "bad weld"), (axes[1], confusing, "good weld")]:
    for ax in row_axes:
        ax.axis("off")
    for ax, (_, case) in zip(row_axes, cases.iterrows()):
        picture = Image.open(pipeline.DATA_DIR / "test" / "images" / case["image"])
        width, height = picture.size
        ax.imshow(picture)
        ax.add_patch(plt.Rectangle(((case["x"] - case["w"] / 2) * width, (case["y"] - case["h"] / 2) * height),
                                   case["w"] * width, case["h"] * height, fill=False, color="#e34948", linewidth=2))
        ax.set_title(f"{kind}, {case['decision']}, score {case['score']:.2f}", fontsize=9)
fig.suptitle("Bad welds the band passes (top) and the good welds it scores highest (bottom); red box: the weld")
fig.tight_layout()
fig.savefig(pipeline.ARTIFACTS_DIR / "failures.png", dpi=120)
show()

## 10. Save what the app needs

The app never trains or predicts on the test set itself. It reads these files
and the weights in `weights/`.

In [ ]:
columns = ["split", "image", "class", "is_bad", "x", "y", "w", "h", "source", "near_duplicate", "score", "score_coco"]
seams[columns].to_csv(pipeline.SEAMS_CSV, index=False)
metrics.to_csv(pipeline.METRICS_CSV, index=False)
cm.to_csv(pipeline.CONFUSION_CSV)
comparison.to_csv(pipeline.ARTIFACTS_DIR / "policy_comparison.csv")
operating_point = {"model": CHOSEN, "threshold": THRESHOLD, "low": LOW, "high": HIGH, "chosen on": "valid",
                   "device": str(DEVICE), "train args": TRAIN_ARGS,
                   "candidates": json.loads(candidates.to_json(orient="records"))}
pipeline.OPERATING_POINT_JSON.write_text(json.dumps(operating_point, indent=2))

# A few test photos for the app's examples, since datasets/ is not committed
examples_dir = pipeline.ARTIFACTS_DIR / "examples"
shutil.rmtree(examples_dir, ignore_errors=True)
examples_dir.mkdir()
test_photos = images[images["split"] == "test"].sort_values("image")
for photos in [test_photos[test_photos["Bad-weld"] > 0], test_photos[test_photos["Bad-weld"] + test_photos["Defect"] == 0]]:
    for name in photos["image"].head(3):
        shutil.copy(pipeline.DATA_DIR / "test" / "images" / name, examples_dir / name)
print("\n".join(sorted(str(p.relative_to(pipeline.ARTIFACTS_DIR)) for p in pipeline.ARTIFACTS_DIR.rglob("*.*"))))

## Download the results

The zip holds what the repo needs, at the same paths as in the repo:

- `weights/best.pt`, the kept model
- `artifacts/`, which the app reads
- the finished training runs, so the repo's `train.ipynb` can redo the
  evaluation without retraining

In [ ]:
import zipfile

from IPython.display import FileLink, display

keep = [pipeline.WEIGHTS_PATH, *pipeline.ARTIFACTS_DIR.rglob("*")]
for name in candidates["run"]:
    run = pipeline.RUNS_DIR / name
    keep += [run / "done.json", run / "args.yaml", run / "results.csv", run / "results.png", run / "weights" / "best.pt"]

os.chdir("/kaggle/working")
with zipfile.ZipFile("weld_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for path in keep:
        if path.is_file():
            z.write(path, path.relative_to(pipeline.ROOT))
print(f"weld_results.zip, {os.path.getsize('weld_results.zip') / 1e6:.1f} MB")
display(FileLink("weld_results.zip"))